In [10]:
from http.client import responses
from typing import List

from anyio.lowlevel import checkpoint
from langchain.agents import create_agent
import os

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.messages.tool import tool_call
from pyexpat.errors import messages

#-----当引入软件包时，_init_才会自动执行
from dotenv import load_dotenv,find_dotenv
env_file=find_dotenv()
print(str(env_file))
load_dotenv(env_file)

DASHSCOPE_API_KEY=os.getenv("DASHSCOPE_API_KEY")

# print(DASHSCOPE_API_KEY)

D:\Zhuxianju\Codefield\CODE_py\pycharm\ChatRAG\heima_agent\.env


## 创建实例化模型

In [ ]:
from langchain_openai import ChatOpenAI
model=ChatOpenAI(
    model="qwen3.6-flash",
    api_key=DASHSCOPE_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    extra_body={
        "enable_thinking": False,  # 关键：关闭思考模式
    },
)

# responses=model.invoke("你好,你是谁")



## 添加tool：TavilySearch

In [ ]:
from langchain_tavily import TavilySearch
from langchain.tools import tool
from pydantic import BaseModel,Field


class Res_refer(BaseModel):
    title:str=Field(description="搜索结果标题")
    url:str=Field(description="搜索结果来源")
class Agent_re_format(BaseModel):
    content:str=Field(description="回答的主体内容")
    refer:List[Res_refer]=Field(description="引用的网页地址，标题")

tavily=TavilySearch(
    max_results=3,
    topic="general"
)

@tool
def search_tool(query:str):
    """从网上搜索信息"""
    return tavily.invoke(query)

## 规划提示词，建立agent对象
-规划提示词

-创建checkpointer对象保存会话历史

In [ ]:
prompt="""
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""
############
#保存会话历史
############
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
# 连接sqlite
# -check_same_thread：检查线程是否同一个，否则报错
connection=sqlite3.connect("resource/checkpointer.db",check_same_thread=False)
checkpointer=SqliteSaver(connection)
checkpointer.setup()    #自动建表

agent=create_agent(
    model=model,
    # system_prompt=prompt,
    checkpointer=checkpointer,
    tools=[search_tool],
    response_format=Agent_re_format,

)

## 创建messages信息,发送给agent

- ***多模态图片信息***

In [ ]:
# multimodal_message = HumanMessage(
#     content=[
#         {"type": "image",
#          "url": "https://img95.699pic.com/photo/50283/5341.jpg_wh860.jpg"},
#         {"type": "text", "text": "我做什么菜好"}
#     ])
# # thread_config4 = {"configurable": {"thread_id": "4"}}
# multimodal_re=agent.stream({"messages":multimodal_message},stream_mode="messages")
# # print(multimodal_re)
# for chunk,metadata in multimodal_re:
#     print(chunk.content,end="")

In [ ]:
# for token, metadata in agent.stream(
#     {"messages": [{"role": "user", "content": "介绍一下湖南张家界"}]},
#     stream_mode="messages"
# ):
#     if token.content:  # Check if there's actual content
#         print(token.content, end="", flush=True)  # Print token

In [ ]:
# message=HumanMessage(content="\"其实剧情很糟糕，奈何作者太嫩了\"是什么梗")
# config={"configurable":{"thread_id":"7-2"}}
# messages=agent.invoke(
#     {"messages":message},
#     config=config,
#     # stream_mode="messages",
# )
# for re in messages["messages"]:
#     re.pretty_print()
